# 读取数据

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from datetime import datetime
from hundredim.util.ml_binning_v8 import Binning

import warnings
warnings.filterwarnings('ignore')

import hundredim
# from hundredim import initialize
from hundredim.util import tools
import hundredim.util.patternAwareScorecardModule as PASM
from hundredim.wrapper.experiment import score_card_experiment
from hundredim.report_generate import score_card_report_render
from hundredim.wrapper import score_card_report
from hundredim.detection import detection
from hundredim.util import analysis_tool
from hundredim.util import tools
tools.set_global_options()
pd.set_option('display.max_rows', None)  # 设置最大显示行数为 None，表示显示所有行
pd.set_option('display.max_columns', None)  # 设置最大显示列数为 None，表示显示所有列

pd.options.display.float_format = '{:.4f}'.format

In [ ]:
df_y = pd.read_csv(r"D:\Data\测试样本\桔子数科\20250303_桔子数科联合建模\桔子数据\14.原始样本\建模_y.csv")
print(df_y.shape)
df_y[:1]

In [ ]:
df_all = pd.read_csv(r"D:\Data\测试样本\桔子数科\20250303_桔子数科联合建模\桔子数据\zf.csv")
print(df_all.shape)
df_all.drop_duplicates(subset=['phone','apply_date'],inplace=True)
print(df_all.shape)
df_all[:1]

In [ ]:
df_all = pd.merge(df_y,df_all,on=['phone','apply_date'],how='inner')
print(df_all.shape)
df_all[:1]

In [ ]:
df_all['gbflag'] =  df_all['mob4']
df_all = df_all[df_all['gbflag'].isin([0,1])]
df_all['gbflag'].value_counts(dropna=False)

In [ ]:
df_all['y'] =df_all['gbflag']
test = df_all.groupby(['dataset','month','y']).size().unstack(fill_value=0)
test['all'] = test[0]+test[1]
test['badrate'] = (test[1]/test['all'])*100
test

In [ ]:
test = df_all.groupby(['dataset','y']).size().unstack(fill_value=0)
test['all'] = test[0]+test[1]
test['badrate'] = (test[1]/test['all'])*100
test

In [ ]:
df_all[:1]

In [ ]:
features = df_all.columns[7:-2].tolist()

In [ ]:
data = df_all.copy()

# 特征筛选

## 特征empty，iv，corr初筛

In [ ]:
import toad 
import numpy as np

selected_data = toad.selection.select(df_all,target='y',empty = 0.99,iv=0.01,corr=0.8,return_drop=True,exclude=base+['gbflag'])
data = selected_data[0]
print(data.shape)
data.head(2)

In [ ]:
features = [x for x in data.columns if x not in (base+['y','gbflag'])]
features

In [ ]:
fea_psi = pd.read_excel(r"D:\Data\测试样本\桔子数科\外部建模\数据质检\al_PSI_channel_month.xlsx")
print(fea_psi.shape)
# fea_psi = fea_psi[fea_psi['渠道']=='nnyx-005u']
print(fea_psi.shape)
fea_psi = fea_psi[fea_psi['PSI']>=0.15]
print(fea_psi.shape)
fea_psi = fea_psi['特征'].unique().tolist()
fea_psi

In [ ]:
fea_iv = pd.read_excel(r"D:\Data\测试样本\桔子数科\外部建模\数据质检\al_IV_channel_month.xlsx")
print(fea_iv.shape)
# fea_iv = fea_iv[fea_iv['渠道']=='nnyx-005u']
print(fea_iv.shape)
has_both = fea_iv.groupby(['特征'])['trend'].apply(lambda x : 
                                                {'increasing','decreasing'}.issubset(set(x)))
fea_iv = has_both[has_both].index.tolist()
fea_iv

In [ ]:
features = [x for x in features if x  not in fea_psi]
features = [x for x in features if x  not in fea_iv]
len(features)

In [ ]:
# for i in features:
#     data[i].fillna(-9999,inplace=True)

In [ ]:
data[features].describe()

# 样本集划分

In [ ]:
data['dataset'].value_counts()

In [ ]:
data.groupby(['dataset','y']).size()

In [ ]:
# data['weight']= data['sampleWeight']
data['weight']= 1
X_train = data[data['dataset']=='train']
odds = max(X_train[X_train['y']==0]['weight'].sum()/X_train[X_train['y']==1]['weight'].sum(),1)
y_train = X_train['y']
X_train_weight = X_train['weight']
X_train = X_train[features]


X_test = data[data['dataset']=='test']
y_test = X_test['y']
X_test_weight = X_test['weight']
X_test = X_test[features]

X_oot = data[data['dataset']=='oot']
y_oot = X_oot['y']
X_oot_weight = X_oot['weight']
X_oot = X_oot[features]

In [ ]:
odds

# 模型训练-贝叶斯调参

In [ ]:
def KS_AUC(pre_score, y_target,save_fig=False,fig_name=None,weights=None):
    import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve, auc

    # 计算 ROC 曲线上的假正率（fpr）、真正率（tpr）和阈值（thr）
    fpr, tpr, thr = roc_curve(y_target, pre_score, sample_weight=weights)
    
    thr[0] = thr[0] -1
    # 计算 AUC 值
    auc = round(auc(fpr, tpr), 3) if round(auc(fpr, tpr), 3) > 0.5 else 1 - round(auc(fpr, tpr), 3)
    
    # 计算 KS 值|
    ks = round(np.abs(tpr - fpr).max(), 3)
    
    fig = plt.figure(constrained_layout=True)
    gs = fig.add_gridspec(2, 1)

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(fpr, tpr, label='ROC Curve (AUC = {})'.format(auc))
    ax1.set_xlabel('False Positive Rate (FPR)')
    ax1.set_ylabel('True Positive Rate (TPR)')
    ax1.set_title('Receiver Operating Characteristic (ROC) Curve')
    ax1.legend(loc='lower right')

    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(thr, tpr, label='True Positive Rate (TPR)')
    ax2.plot(thr, fpr, label='False Positive Rate (FPR)')
    ax2.plot(thr, (tpr - fpr), ls='--', label='KS Value (TPR - FPR)={}'.format(ks), c='green')
    ax2.set_xlabel('Thresholds')
    ax2.set_ylabel('Rate')
    ax2.set_title('KS Curve')
    ax2.legend(loc='upper left')

    if save_fig and fig_name:
        fig.savefig(fig_name, dpi=300)
        plt.close(fig)
    else:
        plt.show()
    
    return ks, auc

## 初筛特征

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_curve
from hyperopt import fmin, hp, tpe, Trials
from hyperopt import STATUS_FAIL, STATUS_OK

# 创建一个空的 DataFrame，用于保存参数组合和结果
results_df = pd.DataFrame(columns=['Params', 'Train KS', 'Train AUC','Test KS','Test AUC','OOT KS','OOT AUC','loss','status'])

# 定义目标优化函数
def objective(params):
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train,sample_weight = X_train_weight)

    y_train_pred = model.predict_proba(X_train)[:, 1]
    y_test_pred = model.predict_proba(X_test)[:, 1]
    y_oot_pred = model.predict_proba(X_oot)[:, 1]
    train_ks, train_auc  = KS_AUC( y_train_pred,y_train,weights = X_train_weight)
    test_ks, test_auc = KS_AUC( y_test_pred,y_test,weights = X_test_weight)
    oot_ks, oot_auc = KS_AUC( y_oot_pred,y_oot,weights = X_oot_weight)
    
    # 将参数组合、KS 和 AUC 值保存到结果 DataFrame 中
    loss = -(oot_ks - 0.5 *np.log (train_ks/oot_ks))
    
    results_df.loc[len(results_df)] = {'Params': params, 'Train KS': train_ks, 'Train AUC': train_auc, 'Test KS': test_ks, 'Test AUC': test_auc,'OOT KS': oot_ks, 'OOT AUC': oot_auc,'loss': loss , 'status': STATUS_OK}
    
    
    return {'loss': loss , 'status': STATUS_OK}

# 定义参数空间
space = {
            'base_score' : 0.5,
            'booster': 'gbtree',
            'objective' : 'binary:logistic',
            'learning_rate': hp.quniform('learning_rate',0.01,0.3,0.01),
            'gamma': hp.quniform('gamma',0,200,2),
            'max_depth': hp.choice('max_depth',[2,3]),
            'subsample': hp.quniform('subsample', 0.5,1,0.1),
            'colsample_bytree': 1,
            'n_estimators': 500,
            'min_child_weight':hp.quniform('min_child_weight',100,1000,50),
            'reg_alpha' : hp.quniform('reg_alpha',0,300,10),
            'reg_lambda' : hp.quniform('reg_lambda', 0,300,10),
            'scale_pos_weight' : hp.quniform('scale_pos_weight',max(int(odds)-2,1),int(odds)+2,1),
            'random_state':2024,
            'tree_method':'gpu_hist',
            'gpu_id': 0,
            'predictor': 'gpu',
            'early_stopping':30,
            'nthread': -1,
            'eval_metric': 'auc'
        }  

# 进行自动调参
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=100, trials=trials)
# 打印最优参数
print("Best parameters:", best)

results_df.to_excel(r'D:\Data\测试样本\桔子数科\外部建模\xgb\hyperopt_results_100_1.xlsx', index=False)


In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_curve
from hyperopt import fmin, hp, tpe, Trials
from hyperopt import STATUS_FAIL, STATUS_OK

# 创建一个空的 DataFrame，用于保存参数组合和结果
results_df = pd.DataFrame(columns=['Params', 'Train KS', 'Train AUC','Test KS','Test AUC','OOT KS','OOT AUC','loss','status'])

# 定义目标优化函数
def objective(params):
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train,sample_weight = X_train_weight)

    y_train_pred = model.predict_proba(X_train)[:, 1]
    y_test_pred = model.predict_proba(X_test)[:, 1]
    y_oot_pred = model.predict_proba(X_oot)[:, 1]
    train_ks, train_auc  = KS_AUC( y_train_pred,y_train,weights = X_train_weight)
    test_ks, test_auc = KS_AUC( y_test_pred,y_test,weights = X_test_weight)
    oot_ks, oot_auc = KS_AUC( y_oot_pred,y_oot,weights = X_oot_weight)
    
    # 将参数组合、KS 和 AUC 值保存到结果 DataFrame 中
    loss = -(oot_ks - 0.5 *np.log (train_ks/oot_ks))
    
    results_df.loc[len(results_df)] = {'Params': params, 'Train KS': train_ks, 'Train AUC': train_auc, 'Test KS': test_ks, 'Test AUC': test_auc,'OOT KS': oot_ks, 'OOT AUC': oot_auc,'loss': loss , 'status': STATUS_OK}
    
    
    return {'loss': loss , 'status': STATUS_OK}

# 定义参数空间
space = {
            'base_score' : 0.5,
            'booster': 'gbtree',
            'objective' : 'binary:logistic',
            'learning_rate': hp.quniform('learning_rate',0.01,0.3,0.01),
            'gamma': hp.quniform('gamma',0,200,2),
            'max_depth': hp.choice('max_depth',[2,3]),
            'subsample': hp.quniform('subsample', 0.5,1,0.1),
            'colsample_bytree': 1,
            'n_estimators': 500,
            'min_child_weight':hp.quniform('min_child_weight',20,1000,10),
            'reg_alpha' : hp.quniform('reg_alpha',0,300,10),
            'reg_lambda' : hp.quniform('reg_lambda', 0,300,10),
            'scale_pos_weight' : hp.quniform('scale_pos_weight',max(int(odds)-2,1),int(odds)+2,1),
            'random_state':2024,
            'tree_method':'hist',
            'early_stopping':30,
            'nthread': -1,
            'eval_metric': 'auc'
        }  

# 进行自动调参
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=100, trials=trials)
# 打印最优参数
print("Best parameters:", best)

results_df.to_excel(r'D:\Data\测试样本\桔子数科\外部建模\xgb\hyperopt_results_100_1.xlsx', index=False)


In [ ]:
best_params = {'base_score': 0.5, 'booster': 'gbtree', 'colsample_bytree': 1, 'early_stopping': 30, 'eval_metric': 'auc', 'gamma': 174.0, 'gpu_id': 0, 'learning_rate': 0.28, 'max_depth': 3, 'min_child_weight': 400.0, 'n_estimators': 500, 'nthread': -1, 'objective': 'binary:logistic', 'predictor': 'gpu', 'random_state': 2024, 'reg_alpha': 240.0, 'reg_lambda': 210.0, 'scale_pos_weight': 22.0, 'subsample': 0.5, 'tree_method': 'gpu_hist'}

In [ ]:
model = xgb.XGBClassifier(**best_params)

model.fit(X_train, y_train,sample_weight = X_train_weight)

In [ ]:
imp_dict = model.get_booster().get_score(importance_type = 'gain')
imp_gain = pd.Series(imp_dict).sort_values(ascending=False)
feat_imp = pd.DataFrame(imp_gain).reset_index()
feat_imp.columns = ['feature','gain']
feat_imp['importance'] = feat_imp['gain'].apply(lambda x : x/feat_imp['gain'].sum())
feat_imp

In [ ]:
import shap
explainer = shap.Explainer(model)
shap_values = explainer(X_train)
shap.plots.bar(shap_values)  # 全局特征重要性（含负值）

In [ ]:
import shap
explainer = shap.Explainer(model)
shap_values = explainer(X_test)
shap.plots.bar(shap_values)  # 全局特征重要性（含负值）

In [ ]:
features = feat_imp.feature[:].tolist()
X_train = X_train[features]
X_test = X_test[features]
X_oot = X_oot[features]

## 根据新特征集合寻参

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_curve
from hyperopt import fmin, hp, tpe, Trials
from hyperopt import STATUS_FAIL, STATUS_OK

# 创建一个空的 DataFrame，用于保存参数组合和结果
results_df = pd.DataFrame(columns=['Params', 'Train KS', 'Train AUC','Test KS','Test AUC','OOT KS','OOT AUC','loss','status'])

# 定义目标优化函数
def objective(params):
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train,sample_weight = X_train_weight)

    y_train_pred = model.predict_proba(X_train)[:, 1]
    y_test_pred = model.predict_proba(X_test)[:, 1]
    y_oot_pred = model.predict_proba(X_oot)[:, 1]
    train_ks, train_auc  = KS_AUC( y_train_pred,y_train,weights = X_train_weight)
    test_ks, test_auc = KS_AUC( y_test_pred,y_test,weights = X_test_weight)
    oot_ks, oot_auc = KS_AUC( y_oot_pred,y_oot,weights = X_oot_weight)
    
    # 将参数组合、KS 和 AUC 值保存到结果 DataFrame 中
    loss = -(oot_ks - 0.5 *np.log (train_ks/oot_ks))
    
    results_df.loc[len(results_df)] = {'Params': params, 'Train KS': train_ks, 'Train AUC': train_auc, 'Test KS': test_ks, 'Test AUC': test_auc,'OOT KS': oot_ks, 'OOT AUC': oot_auc,'loss': loss , 'status': STATUS_OK}
    
    
    return {'loss': loss , 'status': STATUS_OK}

# 定义参数空间
space = {
            'base_score' : 0.5,
            'booster': 'gbtree',
            'objective' : 'binary:logistic',
            'learning_rate': hp.quniform('learning_rate',0.01,0.3,0.01),
            'gamma': hp.quniform('gamma',0,200,2),
            'max_depth': hp.choice('max_depth',[2,3]),
            'subsample': hp.quniform('subsample', 0.5,1,0.1),
            'colsample_bytree': 1,
            'n_estimators': 1000,
            'min_child_weight':hp.quniform('min_child_weight',20,1000,10),
            'reg_alpha' : hp.quniform('reg_alpha',0,300,10),
            'reg_lambda' : hp.quniform('reg_lambda', 0,300,10),
            'scale_pos_weight' : hp.quniform('scale_pos_weight',max(int(odds)-2,1),int(odds)+2,1),
            'random_state':2024,
            'tree_method':'gpu_hist',
            'early_stopping_rounds':30,
            'nthread': -1,
            'eval_metric': 'auc'
        }  

# 进行自动调参
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=100, trials=trials)
# 打印最优参数
print("Best parameters:", best)

results_df.to_excel(r'D:\Data\xgb测试\hyperopt_results_100_nan_1.xlsx', index=False)


## 手动微调参数

In [ ]:
best_params = {'base_score': 0.5, 'booster': 'gbtree', 'colsample_bytree': 1, 'early_stopping_rounds': 30, 'eval_metric': 'auc', 
               'gamma': 62.0, 'learning_rate': 0.02, 'max_depth': 2, 'min_child_weight': 7530.0,
               'n_estimators': 300, 'nthread': -1, 'objective': 'binary:logistic',
               'random_state': 2024, 'reg_alpha': 260.0, 'reg_lambda': 30.0,
               'scale_pos_weight': 45.0, 'subsample': 0.4}

###定版

In [ ]:
best_params = {'base_score': 0.5, 'booster': 'gbtree', 'colsample_bytree': 1, 'early_stopping': 30, 'eval_metric': 'auc', 'gamma': 6.0, 'learning_rate': 0.08, 'max_depth': 3, 'min_child_weight': 170.0, 'n_estimators': 1000, 'nthread': -1, 'objective': 'binary:logistic', 'random_state': 2024, 'reg_alpha': 70.0, 'reg_lambda': 70.0, 'scale_pos_weight': 3.0, 'subsample': 1.0, 'tree_method': 'gpu_hist'}

## nan

In [ ]:
model = xgb.XGBClassifier(**best_params)

model.fit(X_train, y_train,sample_weight = X_train_weight)

In [ ]:
y_train_pred = model.predict_proba(X_train)[:, 1]
y_test_pred = model.predict_proba(X_test)[:, 1]
y_oot_pred = model.predict_proba(X_oot)[:, 1]
# train_ks, train_auc  = KS_AUC( y_train_pred,y_train,save_fig=True,fig_name="train_ks_auc.png")
# test_ks, test_auc = KS_AUC( y_test_pred,y_test,save_fig=True,fig_name="test_ks_auc.png")
# data['weight'] =1
train_ks, train_auc  = KS_AUC( y_train_pred,y_train,weights = X_train_weight)
test_ks, test_auc  = KS_AUC( y_test_pred,y_test,weights = X_test_weight)
oot_ks, oot_auc  = KS_AUC( y_oot_pred,y_oot,weights = X_oot_weight)

# train_ks, train_auc  = KS_AUC( y_train_pred,y_train)
# test_ks, test_auc  = KS_AUC( y_test_pred,y_test)
# oot_ks, oot_auc  = KS_AUC( y_oot_pred,y_oot)

# print(f" TRAIN ks={train_ks},auc={train_auc}\n VALID ks={test_ks},auc={test_auc}")
print(f" TRAIN ks={train_ks},auc={train_auc}\n VALID ks={test_ks},auc={test_auc}\n OOT ks={oot_ks},auc={oot_auc}")

# 模型评估 

## 模型效果

In [ ]:
data['score'] = model.predict_proba(data[features])[:, 0]

In [ ]:
data[:1]

In [ ]:
data['sampleWeight'].value_counts()

In [ ]:
data['weight'].value_counts()

In [ ]:
data[data['channel']=='自营']['weight'].value_counts()

In [ ]:
data['weight'].dtype

In [ ]:
data.loc[data['channel']=='自营','weight_new'] = 1
data.loc[data['channel']=='商户','weight_new'] = 1

In [ ]:
perf_out = PASM.performance_calc(data[data['channel']=='自营'], scores_ls = ['score'],
                                 targets_ls = ['gbflag'], byVars_ls = ['dataset'], weight = 'weight', digits = 4, sv = [])
perf_out['KS_summary'][['dataset', 'score', 'KS']].set_index(['dataset', 'score']).unstack('score')

In [ ]:
perf_out = PASM.performance_calc(data, scores_ls = ['score'],
                                 targets_ls = ['gbflag'], byVars_ls = ['channel','dataset'], weight = 'weight', digits = 4, sv = [])
perf_out['KS_summary'][['channel','dataset', 'score', 'KS']].set_index(['channel','dataset', 'score']).unstack('score')

In [ ]:
perf_out['KS_summary'][['channel','dataset', 'score', 'ROC']].set_index(['channel','dataset', 'score']).unstack('score')

In [ ]:
import matplotlib.pyplot as plt

plt.hist(data['score'],color='skyblue',bins=30)

In [ ]:
def rescaling(ds, score_nm, target_nm, wt_nm):
    import time
    import cvxopt
    import hundredim.util.patternAwareScorecardModule as PASM
    import statsmodels.api as sm
    import math
    X = ds[[score_nm]]
    Y = ds[target_nm]
    wt = ds[wt_nm].apply(lambda x: 1)
    X = sm.add_constant(X)
    glm_mdl = sm.GLM(Y, X, freq_weights = wt, family = sm.families.Binomial())
    glm_r = glm_mdl.fit()
    intercept = glm_r.params[0]
    slope = glm_r.params[1]
    pdo = 50
    intercept_std = math.log(10) - math.log(2)/pdo*800
    slope_std = math.log(2)/pdo
    ds[score_nm + 'rescaled'] = ((intercept-intercept_std)/slope_std +slope/slope_std*ds[score_nm]).round(0)
    return intercept, intercept_std, slope, slope_std

In [ ]:
rescaling(data,'score',target_nm = 'gbflag', wt_nm = 'weight')

In [ ]:
data['gbflag'].replace({0:1,1:0},inplace=True)
data['gbflag'].value_counts()

In [ ]:
data['scorerescaled'].agg([min,max])

In [ ]:
plt.hist(data['scorerescaled'],color='skyblue',bins=50)

In [ ]:
plt.hist(data[data['channel']=='自营']['scorerescaled'],color='skyblue',bins=30)

In [ ]:
plt.hist(data[data['channel']=='商户']['score'],color='skyblue',bins=30)

In [ ]:
import calculate_psi as PSI
import itertools
comb_lst = list(itertools.combinations(np.sort(test['month'].unique()).tolist(), 2))
for i in comb_lst:
    print(i, PSI.calculate_psi(list(test['score'][test['month'] == i[0]]), list(test['score'][test['month'] == i[1]]), bins=20, min_sample=10)[0])

In [ ]:
!pip show itertools

In [ ]:
data['gbflag'] = data['y'].apply(lambda x : 0 if x==1 else 1)
data['gbflag'].value_counts(dropna=False)

In [ ]:
%run "D:\code\ml_binning_v8.py"
Bin = Binning(data[data['dataset']=='train'])
Bin.binning_num('score')

In [ ]:
Bin = Binning(data[data['dataset']=='test'])
Bin.binning_num('score')

In [ ]:
Bin = Binning(data[data['dataset']=='oot'])
Bin.binning_num('score')

## 入模特征明细

In [ ]:
imp_dict = model.get_booster().get_score(importance_type = 'gain')
imp_gain = pd.Series(imp_dict).sort_values(ascending=False)
feat_imp = pd.DataFrame(imp_gain).reset_index()
feat_imp.columns = ['feature','gain']
feat_imp['importance'] = feat_imp['gain'].apply(lambda x : x/feat_imp['gain'].sum())
feat_imp

In [ ]:
features = feat_imp.feature[:].tolist()
X_train = X_train[features]
X_test = X_test[features]
X_oot = X_oot[features]

In [ ]:
model = xgb.XGBClassifier(**best_params)

model.fit(X_train, y_train,sample_weight = X_train_weight)

In [ ]:
y_train_pred = model.predict_proba(X_train)[:, 1]
y_test_pred = model.predict_proba(X_test)[:, 1]
y_oot_pred = model.predict_proba(X_oot)[:, 1]
# train_ks, train_auc  = KS_AUC( y_train_pred,y_train,save_fig=True,fig_name="train_ks_auc.png")
# test_ks, test_auc = KS_AUC( y_test_pred,y_test,save_fig=True,fig_name="test_ks_auc.png")
# data['weight'] =1
train_ks, train_auc  = KS_AUC( y_train_pred,y_train,weights = X_train_weight)
test_ks, test_auc  = KS_AUC( y_test_pred,y_test,weights = X_test_weight)
oot_ks, oot_auc  = KS_AUC( y_oot_pred,y_oot,weights = X_oot_weight)

# train_ks, train_auc  = KS_AUC( y_train_pred,y_train)
# test_ks, test_auc  = KS_AUC( y_test_pred,y_test)
# oot_ks, oot_auc  = KS_AUC( y_oot_pred,y_oot)

# print(f" TRAIN ks={train_ks},auc={train_auc}\n VALID ks={test_ks},auc={test_auc}")
print(f" TRAIN ks={train_ks},auc={train_auc}\n VALID ks={test_ks},auc={test_auc}\n OOT ks={oot_ks},auc={oot_auc}")

In [ ]:
len(model.get_booster().feature_names)

# 模型保存

In [ ]:
import joblib
joblib.dump(model,r"C:\Users\bwfintech\Downloads\浙数抽样样本.part1\A_model.pkl")
# joblib.dump(features,r"features.pkl")

In [ ]:
model.save_model('xgboost_model.model')

# 模型报告


## 1.函数编写

In [ ]:
import pandas as pd
import statsmodels.api as sm
import openpyxl

# 1. 入模型特征的基本情况
def get_feature_statistics(data):
    statistics = data.describe().transpose()
    missing_rate = 1 - data.count() / len(data)
    statistics['Missing Rate'] = missing_rate
    return statistics

# 2. 入模型特征的IV
def calculate_iv(data, target):
    iv_values = []
    for feature in data.columns:
        iv = sm.stats.iv(data[feature], target)
        iv_values.append(iv)
    iv_df = pd.DataFrame({'Feature': data.columns, 'IV': iv_values})
    return iv_df

# 3. 入模型特征的重要性排名
def calculate_feature_importance(data, target,model):
    # 获取特征重要性
    weight = model.get_booster().get_score(importance_type='weight')
    gain = model.get_booster().get_score(importance_type='gain')
    cover = model.get_booster().get_score(importance_type='cover')
    
    # 将字典转换为Series
    weight_series = pd.Series(weight)
    gain_series = pd.Series(gain)
    cover_series = pd.Series(cover)

    importance = pd.DataFrame({'weight': weight_series, 'gain': gain_series, 'cover': cover_series})
    importance['total_gain'] = importance['weight']*importance['gain']
    importance['total_cover'] = importance['weight']*importance['cover']
    importance = importance.sort_values(by='total_gain', ascending=False)
    return importance

In [ ]:
# 单个特征自定义bins计算woe，iv值，逾期率
def calculate_feature_summary(data, feature, y, bins):
    # 计算分箱
    num_bins = bins
    data['bin'], bins = pd.qcut(
    data[feature], num_bins, retbins=True, duplicates='drop')
    data['bin'] = data['bin'].astype(str)

    # 计算每个分箱的样本数量、好用户数量、坏用户数量和坏率
    summary = data.groupby('bin')[y].agg(['count', 'sum'])
    summary.columns = ['total', 'bad']
    summary['good'] = summary['total'] - summary['bad']
    summary['bad_rate'] = summary['bad'] / summary['total']

    # 计算总样本数量、总好用户数量和总坏用户数量
    total_samples = summary['total'].sum()
    total_good = summary['good'].sum()
    total_bad = summary['bad'].sum()

    # 计算WOE值
    summary['woe'] = np.log(
        (summary['good'] / total_good) / (summary['bad'] / total_bad))

    # 计算IV值
    summary['iv'] = (summary['good'] / total_good -
                     summary['bad'] / total_bad) * summary['woe']

    # 输出结果
    summary = summary.reset_index()
    # 在最前面加一列特征名称列
    summary.insert(0, 'feature_name', feature)
    # 添加一行总计数据
    summary_total = summary.agg(['sum'])
    summary_total['feature_name'] = feature
    summary_total['bin'] = '总计'
    summary_total['bad_rate'] = summary_total['bad'] / summary_total['total']
    summary_total['woe'] = ' '
    summary_total = summary_total[['feature_name', 'bin', 'total',
                                   'good', 'bad', 'bad_rate', 'woe', 'iv']].reset_index(drop=True)

    summary_with_total = summary.append(summary_total, ignore_index=True)
    
    return summary_with_total

def sum_feature(data,featurelist,y,bins):
    # 创建空的DataFrame
    result_df = pd.DataFrame()
    for feature in featurelist:
        summary = calculate_feature_summary(data,feature,y,bins)
        result_df = pd.concat([result_df, summary], axis=0)
    return result_df

In [ ]:
# 样本按照0的概率分数降序排列
def model_analy(data,y,prob_0):
    import pandas as pd

    # 计算样本按照0的概率分数降序排列
    data = data.sort_values(by=prob_0, ascending=False).reset_index(drop=True)

    # 将样本按照等分数分组
    n_equal_samples = 10
    samples_per_group = len(data) // n_equal_samples

    groups = pd.cut(range(len(data)), bins=n_equal_samples,
                    labels=range(1, n_equal_samples+1))
    data['group'] = groups
    

    
    # 计算每个分数区间的样本数及占比
    group_summary = data.groupby('group')[prob_0].agg(
        ['count', lambda x: x.count() / len(data)]).reset_index()
    group_summary.columns = ['group', '样本数', '占比']
    


    good_samples = (data[y] == 0)
    bad_samples = (data[y] == 1)
    group_summary['好样本数'] = data[good_samples].groupby('group').size().values
    group_summary['坏样本数'] = data[bad_samples].groupby('group').size().values

    # 计算每个分数区间的逾期率和累计逾期率
    group_summary['逾期率'] = group_summary['坏样本数'] / group_summary['样本数']
    group_summary['累计逾期率'] = group_summary['坏样本数'].cumsum() / \
    group_summary['样本数'].cumsum()
    
    return group_summary


## 2.简单报告生成

In [ ]:
# 创建 Excel 文件
writer = pd.ExcelWriter('模型文件.xlsx', engine='openpyxl')
# 创建工作簿
workbook = writer.book

# 保存入模特征的基本情况
statistics = get_feature_statistics(X)
statistics.to_excel(writer, sheet_name='入模特征的基本情况', index=True)

# 保存特征性排名
featurimportance = calculate_feature_importance(X,y,model)
featurimportance.to_excel(writer, sheet_name='入模特征的重要性排名', index=True)

# 保存特征自定义bins分箱，woe，iv值，逾期率计算
featurelist = list(featurimportance.index)
sum_fea =  sum_feature(data=data,featurelist=featurelist,y='y',bins=10)
sum_fea.to_excel(writer, sheet_name='入模特征的woe，iv，逾期率值情况', index=True)


# 保存样本按照0的概率分数降序排列
data['prob_0'] = model.predict_proba(X)[:, 0]
model_analy =  model_analy(data=data,y='y',prob_0='prob_0')
model_analy.to_excel(writer, sheet_name='样本按0概率降序排列10分箱情况', index=True)


# 保存 Excel 文件
writer.save()

# 关闭 ExcelWriter 和工作簿
writer.close()
workbook.close()

# 模型文件保存

In [ ]:
import pickle
# 保存模型参数pkl文件
model_name_pkl = './model.pkl'
pickle.dump(model, open(model_name_pkl, 'wb'))

# 创建入模特征的 DataFrame
features = X.columns # 假设 X 是特征数据

# 将入模特征保存到一个DataFrame中
df_features = pd.DataFrame(features)

# 保存入模特征的 DataFrame 为 pkl 文件
filename_features = './features.pkl'
df_features.to_pickle(filename_features)

In [ ]:
# 保存模型为 .model 文件
model.save_model('./xgboost.model')

In [ ]:
from nyoka import xgboost_to_pmml
from sklearn.pipeline import Pipeline

# 创建包含模型的 Pipeline
pipeline = Pipeline([
    ('model', model)
])
##  注：特征工程部分未使用转换器内容需自定义函数

# 导出为 PMML 文件
pmml_path = 'model.pmml'
xgboost_to_pmml(pipeline,features,'y',pmml_path)

# 样本打分

## 1.使用.model打分

In [ ]:

# 加载XGBoost模型文件
model = xgb.Booster()
model.load_model("./xgboost.model")

# 加载待打分的样本数据
data_pro = pd.read_csv("./data_processed.csv")


# 提取特征并进行预测
data_pro = data_pro.drop(columns=['y'])
data_pro = data_pro.drop(columns=['ID'])
dtest = xgb.DMatrix(data_pro)
scores = model.predict(dtest)

data_pro['scores_model'] = scores
data_pro.to_excel("样本打分文件_model.xlsx")

## 2.使用pmml文件打分

In [ ]:
from pypmml import Model

# 加载 PMML 文件
pmml_path = 'model.pmml'
pmml_model = Model.fromFile(pmml_path)
# 加载待打分的样本数据
data_pro = pd.read_csv("./data_processed.csv")


# 提取特征并进行预测
data_pro = data_pro.drop(columns=['y'])
data_pro = data_pro.drop(columns=['ID'])


# 预测和打分
predictions = pmml_model.predict(data_pro)

data_pro['scores_pmml'] = predictions['y_probability_1']

data_pro
data_pro.to_excel("样本打分文件_pmml.xlsx")

In [ ]:
!pip list

In [ ]:
!pip install xgboost==1.3.3

In [ ]:
!pip show hyperopt